# Saliency Evaluation: Insertion and Deletion (Rank-1 Identification)

**Insertion** and **Deletion** metrics adapted for **1:N face identification**.

These perturbation-based metrics measure whether the pixels identified as important by a saliency method truly influence the modelâ€™s prediction.

---

## Insertion Metric

The **insertion test** measures how quickly the model recovers its correct prediction when the **most important pixels are progressively added back** to a blank or blurred image.

Procedure:

1. Start from a baseline image (e.g., blurred face).
2. Gradually **insert pixels in order of highest saliency**.
3. After each step, compute the **Rank-1 identification score**.
4. Plot the score vs. percentage of pixels inserted.

A **good explanation** should cause the modelâ€™s score to **increase rapidly** when important pixels are added.

Higher **Area Under the Curve (AUC)** indicates better explanations.

---

## Deletion Metric

The **deletion test** measures how quickly the modelâ€™s prediction deteriorates when **important pixels are removed**.

Procedure:

1. Start with the **original image**.
2. Gradually **remove pixels in order of highest saliency**.
3. Replace them with a baseline value (e.g., blur or mean color).
4. Measure the **Rank-1 identification score** after each step.

A **good explanation** should cause the score to **drop quickly** when important pixels are removed.

Lower **AUC** indicates better explanations.

---

## Rank-1 Identification Setting

Unlike standard classifier evaluation, face recognition operates as a **1:N identification problem**.

For a probe image:

1. Compute its **ArcFace embedding**.
2. Compare it against all **gallery embeddings** using cosine similarity.
3. The identity with the **highest similarity score** is the **Rank-1 prediction**.

During insertion/deletion, we track how the **Rank-1 similarity score to the true identity** changes as pixels are added or removed.





In [11]:
# Cell 1: Imports
import os, json, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Cell 2: Paths + hyperparameters
SPLIT_PATH = "splits/lfw_1N_split.json"
G_IDS_PATH = "cache/gallery_ids.npy"
G_EMB_PATH = "cache/G.npy"

# Evaluation settings
STEPS = 50
INSERTION_BASELINE = "black"   # "blur" or "black"
BLUR_KSIZE = 11

# ---- RISE paths ----
SAL_DIR      = "results/rise_alignedchip_baseline_multi"
OUT_DIR      = os.path.join(SAL_DIR, "eval_margin_auc_multi")
RISE_CSV     = os.path.join(OUT_DIR, f"eval_margin_auc_multi_steps{STEPS}_{INSERTION_BASELINE}.csv")

# ---- CRISE paths ----
CRISE_TAU        = 0.1
CRISE_SAL_DIR    = "results/crise_maps"
CRISE_OUT_DIR    = os.path.join(CRISE_SAL_DIR, "eval_margin_auc")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CRISE_OUT_DIR, exist_ok=True)

print("RISE  SAL_DIR  :", SAL_DIR)
print("RISE  OUT_DIR  :", OUT_DIR)
print("CRISE SAL_DIR  :", CRISE_SAL_DIR)
print("CRISE OUT_DIR  :", CRISE_OUT_DIR)
print("STEPS:", STEPS, "| INSERTION_BASELINE:", INSERTION_BASELINE)

In [ ]:
# Cell 3: Load split + gallery embeddings
with open(SPLIT_PATH, "r") as f:
    split = json.load(f)

gallery = split["gallery"]
probes  = split["probes"]

gallery_ids = np.load(G_IDS_PATH, allow_pickle=True).tolist()
G = np.load(G_EMB_PATH).astype(np.float32)

id_to_index = {pid: i for i, pid in enumerate(gallery_ids)}

def get_gallery_emb(person_id: str) -> np.ndarray:
    e = G[id_to_index[person_id]].astype(np.float32)
    return e / (np.linalg.norm(e) + 1e-12)

# Pre-normalize all gallery vectors so sims = G_unit @ e is cosine
G_unit = G.astype(np.float32)
G_unit /= (np.linalg.norm(G_unit, axis=1, keepdims=True) + 1e-12)

print("Gallery embeddings:", G.shape, "| identities:", len(gallery_ids))

# Identities whose gallery embeddings are zero vectors (face detection failed
# during embedding.ipynb).  Excluded from evaluation so they cannot corrupt
# either s_true (for their own probes) or s_best_other (for all other probes).
BAD_GALLERY_IDS = {"Emile_Lahoud", "John_Paul_II"}
bad_gallery_indices = {id_to_index[pid] for pid in BAD_GALLERY_IDS if pid in id_to_index}
print("Excluded gallery IDs (zero embeddings):", BAD_GALLERY_IDS)

In [14]:
# Cell: Build probe basename -> (true_id, probe_path) lookup
import os, json
import pandas as pd

SPLIT_PATH = "splits/lfw_1N_split.json"

with open(SPLIT_PATH, "r") as f:
    split = json.load(f)

probes  = split["probes"]
gallery = split["gallery"]

probe_stub_to_info = {}  # "Anwar_Ibrahim_0001" -> (true_id, full_path)

for pid, paths in probes.items():
    for p in paths:
        stub = os.path.splitext(os.path.basename(p))[0]
        # stubs should be unique in LFW; if not, keep first and warn later
        if stub not in probe_stub_to_info:
            probe_stub_to_info[stub] = (pid, p)

print("Probe stubs:", len(probe_stub_to_info))

Probe stubs: 7484


In [ ]:
# Cell 4: InsightFace setup (GPU required) + margin_score
# *** Do not run this cell without GPU access. ***
from insightface.app import FaceAnalysis
from rise import build_aligned_chip_112, get_embedding_from_chip

app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]

def margin_score(chip_bgr: np.ndarray, true_id: str) -> float:
    """
    ArcFace 1:N identification margin:
      margin = sim(true_id) - max_{j != true_id, j not in BAD_GALLERY_IDS} sim(j)
    """
    e = get_embedding_from_chip(chip_bgr, rec)
    sims = G_unit @ e
    ti = id_to_index[true_id]
    s_true = float(sims[ti])
    sims_other = sims.copy()
    sims_other[ti] = -1e9
    for bad_idx in bad_gallery_indices:
        sims_other[bad_idx] = -1e9
    return s_true - float(np.max(sims_other))

print("InsightFace ready. Rec model:", type(rec))

In [ ]:
# Cell: Build saliency DataFrames for RISE and CRISE
import re

def rebuild_df(sal_dir: str, prefix: str) -> tuple:
    """
    Scan sal_dir for *_saliency_norm.npy files matching the given prefix.
    Returns (df, unmatched_list).

    prefix examples: "rise_baseline", "crise_tau0p1"
    File naming convention (from rise.py run_name):
        {prefix}_{true_id}_{probe_stub}_N{N}_s{s}_p{p1}_seed{seed}_saliency_norm.npy
    """
    pattern = re.compile(
        rf"^{re.escape(prefix)}_(.+)_N\d+_s\d+_p[\d.]+_seed\d+_saliency_norm\.npy$"
    )

    rows, unmatched = [], []
    for root, _, files in os.walk(sal_dir):
        for fn in sorted(files):
            if not fn.endswith("_saliency_norm.npy"):
                continue
            m = pattern.match(fn)
            if m is None:
                continue   # different prefix — skip silently

            left = m.group(1)   # "{true_id}_{probe_stub}"

            match_info = None
            for stub, (pid, path) in probe_stub_to_info.items():
                if left == pid + "_" + stub:
                    match_info = (pid, path, stub)
                    break

            if match_info is None:
                unmatched.append((fn, "no exact probe stub match"))
                continue

            true_id, probe_path, probe_stub = match_info
            rows.append({
                "true_id":      true_id,
                "probe_stub":   probe_stub,
                "probe_path":   probe_path,
                "saliency_path": os.path.join(root, fn),
            })

    df = (pd.DataFrame(rows)
            .drop_duplicates(subset=["saliency_path"])
            .reset_index(drop=True))
    return df, unmatched


# ---- Load RISE saliency maps ----
rise_df, rise_unmatched = rebuild_df(SAL_DIR, prefix="rise_baseline")
print(f"RISE  saliency maps: {len(rise_df):4d}  |  unmatched: {len(rise_unmatched)}")

# ---- Load CRISE saliency maps (may be empty before crise_run.ipynb runs) ----
tau_str = f"{CRISE_TAU:.3g}".replace(".", "p")
crise_prefix = f"crise_tau{tau_str}"
crise_df, crise_unmatched = rebuild_df(CRISE_SAL_DIR, prefix=crise_prefix)
print(f"CRISE saliency maps: {len(crise_df):4d}  |  unmatched: {len(crise_unmatched)}")
print(f"  (CRISE prefix: '{crise_prefix}')")

In [18]:
# Cell 7: Insertion baseline builder
def make_insertion_baseline(chip_bgr: np.ndarray, mode: str = "blur", ksize: int = 11) -> np.ndarray:
    if mode == "black":
        return np.zeros_like(chip_bgr, dtype=np.uint8)
    if mode == "blur":
        k = int(ksize)
        if k % 2 == 0:
            k += 1
        return cv2.GaussianBlur(chip_bgr, (k, k), 0)
    raise ValueError("mode must be 'blur' or 'black'")

In [ ]:
# Cell: Insertion / Deletion evaluation function
# *** Requires GPU (calls margin_score -> ArcFace inference). ***

def insertion_deletion_curves(chip_bgr, saliency, true_id, steps, baseline_bgr):
    H, W = saliency.shape
    n_pix = H * W
    order = np.argsort(saliency.reshape(-1))[::-1]

    del_img = chip_bgr.copy()
    ins_img = baseline_bgr.copy()

    frac = np.linspace(0.0, 1.0, steps + 1, dtype=np.float32)
    del_scores  = [margin_score(del_img, true_id)]
    ins_scores  = [margin_score(ins_img, true_id)]

    for t in range(1, steps + 1):
        k = int(round(float(frac[t]) * n_pix))
        idx = order[:k]
        ys, xs = idx // W, idx % W
        del_img[ys, xs] = 0
        ins_img[ys, xs] = chip_bgr[ys, xs]
        del_scores.append(margin_score(del_img, true_id))
        ins_scores.append(margin_score(ins_img, true_id))

    return frac, np.array(del_scores, np.float32), np.array(ins_scores, np.float32)


def run_eval(df: pd.DataFrame, out_dir: str, label: str = "") -> pd.DataFrame:
    """
    Run insertion/deletion evaluation on every probe in df.
    Saves per-probe CSV to out_dir.
    Returns eval_df with columns: true_id, probe_path, saliency_path,
      del_auc_margin, ins_auc_margin, margin_clean, margin_deleted_all,
      margin_base, margin_inserted_all.
    Also saves curve arrays as .npz for later plotting.
    """
    rows, all_del, all_ins, frac_ref = [], [], [], None
    t0 = time.time()

    for i, r in df.iterrows():
        true_id   = r["true_id"]
        probe_path = r["probe_path"]
        sal_path   = r["saliency_path"]

        if true_id in BAD_GALLERY_IDS:
            print(f"[{i+1}/{len(df)}] SKIP | {true_id} | zero gallery embedding")
            continue

        try:
            img = cv2.imread(probe_path)
            if img is None:
                raise ValueError(f"Could not read {probe_path}")

            chip_bgr = build_aligned_chip_112(img, app)
            baseline_bgr = make_insertion_baseline(chip_bgr, mode=INSERTION_BASELINE,
                                                   ksize=BLUR_KSIZE)

            sal = np.load(sal_path).astype(np.float32)
            if sal.shape != (112, 112):
                raise ValueError(f"Bad sal shape {sal.shape}")

            frac, del_sc, ins_sc = insertion_deletion_curves(
                chip_bgr, sal, true_id, STEPS, baseline_bgr
            )

            del_auc = float(np.trapz(del_sc, frac))
            ins_auc = float(np.trapz(ins_sc, frac))

            rows.append(dict(
                true_id=true_id, probe_path=probe_path, saliency_path=sal_path,
                del_auc_margin=del_auc, ins_auc_margin=ins_auc,
                margin_clean=float(del_sc[0]),
                margin_deleted_all=float(del_sc[-1]),
                margin_base=float(ins_sc[0]),
                margin_inserted_all=float(ins_sc[-1]),
            ))

            if frac_ref is None:
                frac_ref = frac.copy()
            all_del.append(del_sc)
            all_ins.append(ins_sc)

            print(f"[{i+1}/{len(df)}] {label} | {true_id} "
                  f"| del={del_auc:.4f} ins={ins_auc:.4f}")

        except Exception as e:
            print(f"[{i+1}/{len(df)}] FAIL | {true_id} | {repr(e)}")

    elapsed = time.time() - t0
    eval_df = pd.DataFrame(rows)

    csv_path = os.path.join(out_dir,
        f"eval_margin_auc_steps{STEPS}_{INSERTION_BASELINE}.csv")
    eval_df.to_csv(csv_path, index=False)
    print(f"\n{label} done: {len(eval_df)} probes | {elapsed/60:.1f} min | saved {csv_path}")

    if all_del:
        npz_path = os.path.join(out_dir,
            f"curves_steps{STEPS}_{INSERTION_BASELINE}.npz")
        np.savez(npz_path,
                 frac=frac_ref,
                 del_curves=np.stack(all_del),
                 ins_curves=np.stack(all_ins))
        print(f"Curves saved: {npz_path}")

    return eval_df

In [ ]:
# Cell: Run RISE eval (load existing CSV if available; else run — requires GPU)
_rise_csv_legacy = os.path.join(OUT_DIR,
    f"eval_margin_auc_multi_steps{STEPS}_{INSERTION_BASELINE}.csv")
_rise_csv_new    = os.path.join(OUT_DIR,
    f"eval_margin_auc_steps{STEPS}_{INSERTION_BASELINE}.csv")

def _try_load_csv(path):
    """Load CSV, return DataFrame or None if missing/empty/corrupt."""
    if not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path)
        if df.empty or len(df.columns) == 0:
            print(f"WARNING: {path} is empty — will re-run.")
            return None
        return df
    except Exception as e:
        print(f"WARNING: could not read {path} ({e}) — will re-run.")
        return None

rise_eval_df = _try_load_csv(_rise_csv_legacy)
if rise_eval_df is not None:
    print(f"Loaded existing RISE eval CSV ({len(rise_eval_df)} rows): {_rise_csv_legacy}")
else:
    rise_eval_df = _try_load_csv(_rise_csv_new)
    if rise_eval_df is not None:
        print(f"Loaded existing RISE eval CSV ({len(rise_eval_df)} rows): {_rise_csv_new}")
    else:
        print("Running RISE eval (requires GPU)...")
        rise_eval_df = run_eval(rise_df, OUT_DIR, label="RISE")

rise_eval_df.head()

In [ ]:
# Cell: Run CRISE eval (requires GPU)
# *** Run after crise_run.ipynb has populated results/crise_maps/ ***
print(f"CRISE saliency maps available: {len(crise_df)}")

if len(crise_df) == 0:
    print("No CRISE maps found. Run crise_run.ipynb first.")
    crise_eval_df = pd.DataFrame()
else:
    crise_eval_df = run_eval(crise_df, CRISE_OUT_DIR, label="CRISE")
    crise_eval_df.head()

In [ ]:
# Cell: Comparison table + sanity checks
if crise_eval_df.empty:
    print("CRISE eval not yet run — skip comparison.")
else:
    summary = pd.DataFrame([
        {
            "Method":           "RISE (baseline)",
            "n_probes":         len(rise_eval_df),
            "Deletion AUC":     rise_eval_df["del_auc_margin"].mean(),
            "Insertion AUC":    rise_eval_df["ins_auc_margin"].mean(),
            "Mean margin clean": rise_eval_df["margin_clean"].mean(),
        },
        {
            "Method":           f"CRISE (tau={CRISE_TAU})",
            "n_probes":         len(crise_eval_df),
            "Deletion AUC":     crise_eval_df["del_auc_margin"].mean(),
            "Insertion AUC":    crise_eval_df["ins_auc_margin"].mean(),
            "Mean margin clean": crise_eval_df["margin_clean"].mean(),
        },
    ])
    print(summary.to_string(index=False))

    # ---- Sanity checks ----
    rise_del  = rise_eval_df["del_auc_margin"].mean()
    rise_ins  = rise_eval_df["ins_auc_margin"].mean()
    crise_del = crise_eval_df["del_auc_margin"].mean()
    crise_ins = crise_eval_df["ins_auc_margin"].mean()

    print("\n--- Sanity checks ---")
    ok_ins = crise_ins > rise_ins
    ok_del = crise_del < rise_del
    print(f"CRISE insertion AUC ({crise_ins:.4f}) > RISE ({rise_ins:.4f}): "
          f"{'PASS ✓' if ok_ins else 'FAIL ✗ — revisit tau or weighting'}")
    print(f"CRISE deletion  AUC ({crise_del:.4f}) < RISE ({rise_del:.4f}): "
          f"{'PASS ✓' if ok_del else 'FAIL ✗ — revisit tau or weighting'}")

    if not (ok_ins and ok_del):
        print("\nWARNING: One or both sanity checks failed.")
        print("Actions: (1) check tau ablation [0.05, 0.1, 0.2, 0.5], "
              "(2) verify crise_maps/ used the correct weight_fn.")

    # ---- Save comparison summary ----
    cmp_path = os.path.join(CRISE_OUT_DIR, "rise_vs_crise_comparison.csv")
    summary.to_csv(cmp_path, index=False)
    print(f"\nSaved: {cmp_path}")

In [ ]:
# Cell: Stability test — CRISE map variance across two random seeds
# *** Requires GPU. Run after crise_run.ipynb completes. ***
#
# For each of STABILITY_N probes, run CRISE a second time with a different seed.
# Report cosine similarity and L1 distance between the two maps.
# High cosine similarity (> ~0.9) = stable estimator.

import importlib
import rise, crise
importlib.reload(rise); importlib.reload(crise)
from crise import run_crise, TAU

STABILITY_N = 20
STABILITY_SEED_OFFSET = 999_999
STABILITY_DIR = os.path.join(CRISE_SAL_DIR, "stability")
os.makedirs(STABILITY_DIR, exist_ok=True)

# Load the CRISE run summary to get original seeds
_crise_summary_candidates = [
    f for f in os.listdir(CRISE_SAL_DIR)
    if f.startswith("summary_crise") and f.endswith(".csv")
]

if not _crise_summary_candidates or crise_df.empty:
    print("CRISE summary CSV not found or crise_df empty — run crise_run.ipynb first.")
else:
    crise_summary = pd.read_csv(
        os.path.join(CRISE_SAL_DIR, _crise_summary_candidates[0])
    )

    # Pick probes that completed successfully
    stable_probes = (
        crise_summary[crise_summary["saliency_path"].notna()]
        .head(STABILITY_N)
        .reset_index(drop=True)
    )
    print(f"Running stability test on {len(stable_probes)} probes...")

    stab_rows = []
    for _, row in stable_probes.iterrows():
        true_id    = row["true_id"]
        probe_path = row["probe_path"]
        run_seed_a = int(row["run_seed"])
        sal_path_a = row["saliency_path"]

        # Second run with offset seed
        run_seed_b = run_seed_a + STABILITY_SEED_OFFSET
        out_b = run_crise(
            true_id=true_id,
            probe_path=probe_path,
            G=G, id_to_index=id_to_index,
            rec=rec, app=app,
            run_seed=run_seed_b,
            out_dir=STABILITY_DIR,
            tau=TAU,
        )
        sal_path_b = out_b["saliency_path"]

        sal_a = np.load(sal_path_a).flatten().astype(np.float64)
        sal_b = np.load(sal_path_b).flatten().astype(np.float64)

        cos_sim = float(
            np.dot(sal_a, sal_b)
            / (np.linalg.norm(sal_a) * np.linalg.norm(sal_b) + 1e-12)
        )
        l1 = float(np.mean(np.abs(sal_a - sal_b)))

        stab_rows.append(dict(true_id=true_id, cosine_similarity=cos_sim, l1_distance=l1))
        print(f"  {true_id:40s}  cos={cos_sim:.4f}  L1={l1:.4f}")

    stab_df = pd.DataFrame(stab_rows)
    print(f"\nStability summary (n={len(stab_df)}):")
    print(f"  Mean cosine similarity : {stab_df['cosine_similarity'].mean():.4f}")
    print(f"  Std  cosine similarity : {stab_df['cosine_similarity'].std():.4f}")
    print(f"  Mean L1 distance       : {stab_df['l1_distance'].mean():.4f}")

    stab_csv = os.path.join(CRISE_OUT_DIR, "stability_test.csv")
    stab_df.to_csv(stab_csv, index=False)
    print(f"\nSaved: {stab_csv}")

In [ ]:
# Cell: Mean curve plot — RISE vs CRISE side by side
# (Runs without GPU; loads pre-saved .npz curve arrays)

def load_curves(eval_dir, steps, baseline):
    npz = os.path.join(eval_dir, f"curves_steps{steps}_{baseline}.npz")
    if not os.path.exists(npz):
        return None, None, None
    d = np.load(npz)
    return d["frac"], d["del_curves"], d["ins_curves"]

frac_r, del_r, ins_r = load_curves(OUT_DIR,       STEPS, INSERTION_BASELINE)
frac_c, del_c, ins_c = load_curves(CRISE_OUT_DIR, STEPS, INSERTION_BASELINE)

if frac_r is None and frac_c is None:
    print("No curve .npz files found — run eval cells first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    for ax, frac, del_curves, ins_curves, label in [
        (axes[0], frac_r, del_r, ins_r, "RISE"),
        (axes[1], frac_c, del_c, ins_c, f"CRISE (τ={CRISE_TAU})"),
    ]:
        if frac is None:
            ax.set_title(f"{label} — no data")
            continue

        del_mean = del_curves.mean(axis=0)
        ins_mean = ins_curves.mean(axis=0)
        del_auc  = float(np.trapz(del_mean, frac))
        ins_auc  = float(np.trapz(ins_mean, frac))

        ax.plot(frac, del_mean, label=f"Deletion  (AUC={del_auc:.3f})")
        ax.plot(frac, ins_mean, label=f"Insertion (AUC={ins_auc:.3f})")
        ax.axhline(0, color="k", linewidth=0.6, linestyle="--")
        ax.set_xlabel("Fraction of pixels modified")
        ax.set_ylabel("Identification margin")
        ax.set_title(f"{label}  (n={del_curves.shape[0]})")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle("Mean insertion / deletion curves — RISE vs CRISE", fontsize=12)
    plt.tight_layout()

    fig_path = os.path.join(CRISE_OUT_DIR, f"rise_vs_crise_curves_{INSERTION_BASELINE}.png")
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()